# Dataset 7: robustness to noise level

This notebook tests how DCIts behaves as the noise standard deviation $\sigma$ increases in Dataset 7.
For each value of $\sigma$, the model is trained multiple times and the notebook reports:

- prediction error on the test set;
- correlation between estimated and ground-truth alpha coefficients;
- stability of selected interpretable coefficients across noise levels.


In [ ]:
import json
import os
import pickle
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Load seminar-specific utilities and original DCIts source.
# This supports two layouts:
# 1. notebook inside DCIts/examples, where DCIts/src contains the utilities;
# 2. this inspection folder, where seminar-specific utilities are under
#    seminar_inspection/support_utils/src and the original DCIts clone is nearby.
def add_dcits_to_path():
    cwd = Path.cwd().resolve()
    search_bases = [cwd, *cwd.parents]

    utility_root = None
    dcits_root = None

    for base in search_bases:
        candidates = [
            base,
            base / "DCIts",
            base / "support_utils",
            base / "seminar_inspection" / "support_utils",
        ]
        for candidate in candidates:
            if utility_root is None and (candidate / "src" / "utils.py").exists():
                utility_root = candidate
            if dcits_root is None and (candidate / "src" / "dcits.py").exists():
                dcits_root = candidate

    if utility_root is None:
        raise FileNotFoundError("Could not find src/utils.py.")
    if dcits_root is None:
        raise FileNotFoundError("Could not find src/dcits.py from the original DCIts project.")

    # Insert DCIts first, then seminar utilities, so support_utils/src/utils.py
    # takes precedence over the original DCIts/src/utils.py while src.dcits
    # remains available through the namespace-style src package.
    for candidate in [str(dcits_root), str(utility_root)]:
        if candidate in sys.path:
            sys.path.remove(candidate)
        sys.path.insert(0, candidate)

    return utility_root, dcits_root


UTILITY_ROOT, DCITS_ROOT = add_dcits_to_path()
print(f"Using seminar utilities from: {UTILITY_ROOT}")
print(f"Using DCIts source from: {DCITS_ROOT}")
from src.utils import *

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
})

if torch.cuda.is_available():
    device = torch.device("cuda:0")
    print(f"Using {torch.cuda.get_device_name(device)}")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU instead.")

ARTIFACT_ROOT = Path("artifacts")
RUN_NAME = "noise_sigma"
OUT_DIR = ARTIFACT_ROOT / RUN_NAME
DATA_DIR = OUT_DIR / "data"
FIG_DIR = OUT_DIR / "figures"

for path in (DATA_DIR, FIG_DIR):
    path.mkdir(parents=True, exist_ok=True)


## Dataset 7 ground truth

The Dataset 7 generating equations are

$$
\begin{split}
X_{1,t} &= \alpha^\text{gt}_{1,1,1} X_{1,t-1} + \alpha^\text{gt}_{1,1,5} X_{1,t-5} + \epsilon_{1,t}, \\
X_{2,t} &= 1 + \alpha^\text{gt}_{2,1,2} X_{1,t-2} + \epsilon_{2,t}, \\
X_{3,t} &= \alpha^\text{gt}_{3,2,1} X_{2,t-1} + \alpha^\text{gt}_{3,4,4} X_{4,t-4} + \epsilon_{3,t}, \\
X_{4,t} &= 1 + \alpha^\text{gt}_{4,3,4} X_{3,t-4} + \alpha^\text{gt}_{4,5,1} X_{5,t-1} + \epsilon_{4,t}, \\
X_{5,t} &= \alpha^\text{gt}_{5,5,4} X_{5,t-4} + \alpha^\text{gt}_{5,2,1} X_{2,t-1} + \epsilon_{5,t}.
\end{split}
$$


In [ ]:
# Reproducibility
seed = 1000
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Noise sweep
sigma_values = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0, 2.0, 3.0]
noise_frequency = 1.0
mean = 0.0

# Data and DCIts settings
ts_length = 20000
burn_in = 1000
n_runs = 5
window_length_gp = 5
temperature = 1.0
order = [1, 1]

# Dataset dimensions
no_of_timeseries_gp = 5
dataset_name = "Dataset 7"
series_names = [f"X{i}" for i in range(1, no_of_timeseries_gp + 1)]

# Ground-truth tensor: [target, source, lag_index], where lag_index 0 means lag 1.
ground_truth_alpha = torch.zeros(no_of_timeseries_gp, no_of_timeseries_gp, window_length_gp)
ground_truth_alpha[0, 0, 0] = 1 / 4
ground_truth_alpha[0, 0, 4] = 3 / 4
ground_truth_alpha[1, 0, 1] = -1
ground_truth_alpha[2, 1, 0] = 1
ground_truth_alpha[2, 3, 3] = 1
ground_truth_alpha[3, 2, 3] = -2 / 7
ground_truth_alpha[3, 4, 0] = -5 / 7
ground_truth_alpha[4, 4, 3] = 12 / 22
ground_truth_alpha[4, 1, 0] = 10 / 22

alpha_mask = (ground_truth_alpha != 0).float()
ground_truth_bias = torch.tensor([0, 1, 0, 1, 0])

gt_alphas = [
    (0, 0, 0, 1 / 4),
    (0, 0, 4, 3 / 4),
    (1, 0, 1, -1),
    (2, 1, 0, 1),
    (2, 3, 3, 1),
    (3, 2, 3, -2 / 7),
    (3, 4, 0, -5 / 7),
    (4, 4, 3, 12 / 22),
    (4, 1, 0, 10 / 22),
]

CACHE_PATH = DATA_DIR / f"DCIts_DS7_L{window_length_gp}_seed{seed}_ts{ts_length}_runs{n_runs}.pkl"
print("Cache:", CACHE_PATH)


In [ ]:
def save_pickle(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)


def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def make_train_config():
    return {
        "verbose": False,
        "device": device,
        "learning_rate": 1e-3,
        "scheduler_patience": 5,
        "early_stopping_modifier": 2,
        "criterion": nn.MSELoss(),
    }


def alpha_corr_with_ground_truth(stats):
    # DCIts stores the model lag axis reversed relative to the ground-truth tensor.
    estimated = np.flip(to_numpy(stats["alpha"][1]["mean"]), axis=2)
    truth = to_numpy(ground_truth_alpha)
    return np.corrcoef(estimated.flatten(), truth.flatten())[0, 1]


def flip_lag(lag):
    return window_length_gp - lag - 1


def stable_ylim(ymin, ymax, min_range=0.5):
    ymin = float(ymin)
    ymax = float(ymax)
    if ymax < ymin:
        ymin, ymax = ymax, ymin
    if ymax - ymin < min_range:
        mid = 0.5 * (ymin + ymax)
        ymin = mid - 0.5 * min_range
        ymax = mid + 0.5 * min_range
    margin = 0.06 * (ymax - ymin)
    return ymin - margin, ymax + margin


## Run the sigma sweep

Set `FORCE_RECOMPUTE = True` if you want to ignore the cache and retrain all models.


In [ ]:
FORCE_RECOMPUTE = False
PLOT_TIME_SERIES = False

results_by_sigma = {}
if CACHE_PATH.exists() and not FORCE_RECOMPUTE:
    cache = load_pickle(CACHE_PATH)
    results_by_sigma = cache.get("results_by_sigma", {})
    print(f"Loaded {len(results_by_sigma)} cached sigma values.")

for sigma in sigma_values:
    sigma_key = float(sigma)
    if sigma_key in results_by_sigma and not FORCE_RECOMPUTE:
        print(f"Skipping sigma={sigma_key}: cached.")
        continue

    print("\n" + "#" * 80)
    print(f"sigma={sigma_key}")

    time_series = dataset(
        ts_length,
        ground_truth_alpha,
        ground_truth_bias,
        noise_frequency=noise_frequency,
        mu=mean,
        sigma=sigma_key,
        seed=seed,
    )

    if PLOT_TIME_SERIES:
        plot_ts(time_series, dataset_name=f"{dataset_name} (sigma={sigma_key})", alpha=0.7)

    results = collect_multiple_runs(
        n_runs=n_runs,
        time_series=time_series,
        window_size=window_length_gp,
        temperature=temperature,
        order=order,
        config=make_train_config(),
        seed=seed,
        verbose=False,
    )
    stats = calculate_multiple_run_statistics(results)

    rmse = float(np.sqrt(results["summary"]["mean_test_loss"]))
    alpha_corr = float(alpha_corr_with_ground_truth(stats))

    results_by_sigma[sigma_key] = {
        "stats": stats,
        "rmse": rmse,
        "alpha_corr": alpha_corr,
        "summary": results.get("summary", {}),
        "timestamp": time.time(),
    }

    save_pickle({
        "meta": {
            "dataset_name": dataset_name,
            "seed": seed,
            "ts_length": ts_length,
            "burn_in": burn_in,
            "noise_frequency": noise_frequency,
            "mean": mean,
            "window_length": window_length_gp,
            "n_runs": n_runs,
        },
        "sigma_values": sigma_values,
        "ground_truth_alpha": to_numpy(ground_truth_alpha),
        "ground_truth_bias": to_numpy(ground_truth_bias),
        "results_by_sigma": results_by_sigma,
    }, CACHE_PATH)

    print(f"sigma={sigma_key}: RMSE={rmse:.6f}, alpha_corr={alpha_corr:.3f}")

available_sigmas = [float(s) for s in sigma_values if float(s) in results_by_sigma]
stats_by_sigma = [results_by_sigma[s]["stats"] for s in available_sigmas]
rmse_values = [results_by_sigma[s]["rmse"] for s in available_sigmas]
alpha_corr_values = [results_by_sigma[s]["alpha_corr"] for s in available_sigmas]

summary_sigma = pd.DataFrame({
    "sigma": available_sigmas,
    "rmse": rmse_values,
    "alpha_corr": alpha_corr_values,
})
summary_sigma.to_csv(DATA_DIR / "noise_sigma_summary.csv", index=False)
summary_sigma


## Summary plots


In [ ]:
def plot_rmse_vs_sigma(summary):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(summary["sigma"], summary["rmse"], marker="o", linewidth=2)
    ax.set_xlabel(r"noise standard deviation $\sigma$")
    ax.set_ylabel("RMSE")
    ax.set_title("DCIts robustness to noise variance")
    ax.set_ylim(0, max(float(summary["rmse"].max()) * 1.08, 0.05))
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "RMSE_vs_sigma.png", bbox_inches="tight", dpi=300)
    fig.savefig(FIG_DIR / "RMSE_vs_sigma.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)


def plot_alpha_corr_vs_sigma(summary):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(summary["sigma"], summary["alpha_corr"], marker="o", linewidth=2, color="tab:orange")
    ax.set_xlabel(r"noise standard deviation $\sigma$")
    ax.set_ylabel(r"corr($\hat{\alpha}$, $\alpha^{gt}$)")
    ax.set_title("Alpha correlation versus noise")
    ax.set_ylim(-1.05, 1.05)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "alpha_corr_vs_sigma.png", bbox_inches="tight", dpi=300)
    fig.savefig(FIG_DIR / "alpha_corr_vs_sigma.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)


plot_rmse_vs_sigma(summary_sigma)
plot_alpha_corr_vs_sigma(summary_sigma)


In [ ]:
def plot_alpha_grouped(stats_list, sigma_values, gt_alphas, min_y_range=0.5):
    grouped = {}
    for target, source, lag, gt_val in gt_alphas:
        grouped.setdefault(source, []).append((target, source, lag, gt_val))

    for source in sorted(grouped):
        fig, ax = plt.subplots(figsize=(9, 5))
        all_mins = []
        all_maxs = []

        for target, source, lag, gt_val in grouped[source]:
            lag_model = flip_lag(lag)
            means = np.array([stats["alpha"][1]["mean"][target, source, lag_model] for stats in stats_list], dtype=float)
            stds = np.array([stats["alpha"][1]["std"][target, source, lag_model] for stats in stats_list], dtype=float)
            label = rf"$\alpha_{{{target+1},{source+1},{lag+1}}}$"

            ax.errorbar(sigma_values, means, yerr=stds, marker="o", capsize=3, linewidth=1.6, label=label)
            ax.axhline(gt_val, color="black", linestyle="--", linewidth=1.2, alpha=0.6)

            all_mins.extend([np.min(means - stds), gt_val])
            all_maxs.extend([np.max(means + stds), gt_val])

        ax.set_ylim(*stable_ylim(min(all_mins), max(all_maxs), min_range=min_y_range))
        ax.set_title(f"Alpha estimates from {series_names[source]}")
        ax.set_xlabel(r"noise standard deviation $\sigma$")
        ax.set_ylabel(r"$\alpha$ mean +/- SD")
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=2, frameon=True)
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"{series_names[source]}_alpha_vs_sigma.png", bbox_inches="tight", dpi=300)
        fig.savefig(FIG_DIR / f"{series_names[source]}_alpha_vs_sigma.pdf", bbox_inches="tight")
        plt.show()
        plt.close(fig)


def plot_alpha_error_grouped(stats_list, sigma_values, gt_alphas):
    grouped = {}
    for target, source, lag, gt_val in gt_alphas:
        grouped.setdefault(source, []).append((target, source, lag, gt_val))

    for source in sorted(grouped):
        fig, ax = plt.subplots(figsize=(9, 5))
        max_error = 0.0

        for target, source, lag, gt_val in grouped[source]:
            lag_model = flip_lag(lag)
            errors = []
            for stats in stats_list:
                est = stats["alpha"][1]["mean"][target, source, lag_model]
                errors.append(abs(est - gt_val))
            errors = np.array(errors)
            max_error = max(max_error, float(errors.max()))
            ax.plot(sigma_values, errors, marker="o", linewidth=1.6, label=rf"$\alpha_{{{target+1},{source+1},{lag+1}}}$")

        ax.set_ylim(0, max(max_error * 1.08, 0.05))
        ax.set_title(f"Alpha error from {series_names[source]}")
        ax.set_xlabel(r"noise standard deviation $\sigma$")
        ax.set_ylabel("absolute alpha error")
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=2, frameon=True)
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"{series_names[source]}_alpha_error_vs_sigma.png", bbox_inches="tight", dpi=300)
        fig.savefig(FIG_DIR / f"{series_names[source]}_alpha_error_vs_sigma.pdf", bbox_inches="tight")
        plt.show()
        plt.close(fig)


plot_alpha_grouped(stats_by_sigma, available_sigmas, gt_alphas)
plot_alpha_error_grouped(stats_by_sigma, available_sigmas, gt_alphas)


In [ ]:
def plot_alpha_all_in_one(stats_list, sigma_values, gt_alphas, min_y_range=0.5):
    fig, ax = plt.subplots(figsize=(12, 6))
    all_mins = []
    all_maxs = []

    for target, source, lag, gt_val in gt_alphas:
        lag_model = flip_lag(lag)
        means = np.array([stats["alpha"][1]["mean"][target, source, lag_model] for stats in stats_list], dtype=float)
        stds = np.array([stats["alpha"][1]["std"][target, source, lag_model] for stats in stats_list], dtype=float)
        label = rf"$\alpha_{{{target+1},{source+1},{lag+1}}}$"

        err = ax.errorbar(sigma_values, means, yerr=stds, marker="o", capsize=3, linewidth=1.6, label=label)
        ax.axhline(gt_val, color=err.lines[0].get_color(), linestyle="--", linewidth=1.2, alpha=0.6)

        all_mins.extend([np.min(means - stds), gt_val])
        all_maxs.extend([np.max(means + stds), gt_val])

    ax.set_ylim(*stable_ylim(min(all_mins), max(all_maxs), min_range=min_y_range))
    ax.set_title("Alpha estimates: all active links")
    ax.set_xlabel(r"noise standard deviation $\sigma$")
    ax.set_ylabel(r"$\alpha$ mean +/- SD")
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2, frameon=True)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "ALL_alpha_vs_sigma.png", bbox_inches="tight", dpi=300)
    fig.savefig(FIG_DIR / "ALL_alpha_vs_sigma.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)


plot_alpha_all_in_one(stats_by_sigma, available_sigmas, gt_alphas)


## Optional heatmaps for low and high noise

These heatmaps are useful as representative examples. They compare estimated alpha against the known ground-truth tensor for the lowest and highest available noise levels.


In [ ]:
def plot_representative_heatmaps(sigmas_to_plot=None):
    if sigmas_to_plot is None:
        sigmas_to_plot = [available_sigmas[0], available_sigmas[-1]]

    for sigma in sigmas_to_plot:
        stats = results_by_sigma[float(sigma)]["stats"]
        print(f"sigma={sigma}")
        plot_alphas(
            stats["alpha"][1]["mean"],
            ground_truth_alpha,
            figsize=(8, 7),
            font_size=11,
            space=0.0,
        )


plot_representative_heatmaps()
